# 03 - Generación de Embeddings

Genera representaciones vectoriales (embeddings) de las letras de canciones usando `sentence-transformers`.
Los embeddings se guardan en su forma cruda (384 dims) para que el siguiente paso del pipeline
pueda determinar empíricamente la reducción de dimensionalidad óptima.

**Entrada**: `data/processed/train.parquet`, `data/processed/test.parquet`  
**Salida**: `data/embeddings/train_embeddings_raw.parquet`, `data/embeddings/test_embeddings_raw.parquet`

---

**CONFIGURACIÓN**: Ajustá `USE_SUBSET` según tus necesidades.
- `True` → 50K samples estratificados (~10-15 min). Útil para desarrollo y calibración.
- `False` → Dataset completo (~545K, ~2-3 horas). Para resultados finales.

## 1. Importación de Librerías

In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
import time
from pathlib import Path

print('✓ Librerías importadas')

✓ Librerías importadas


## 2. Configuración

In [2]:
# ============================================================
# CONFIGURACIÓN
# ============================================================
USE_SUBSET = False  # True = 50K | False = dataset completo
SUBSET_SIZE = 50000
# ============================================================

DATA_PROCESSED  = Path('../data/processed')
DATA_EMBEDDINGS = Path('../data/embeddings')
DATA_EMBEDDINGS.mkdir(parents=True, exist_ok=True)

suffix = '_subset' if USE_SUBSET else ''

print('=' * 55)
print('CONFIGURACIÓN')
print('=' * 55)
if USE_SUBSET:
    print(f'  Modo   : SUBSET ({SUBSET_SIZE:,} samples estratificados)')
    print(f'  Tiempo : ~10-15 minutos')
else:
    print(f'  Modo   : DATASET COMPLETO (~545K samples)')
    print(f'  Tiempo : ~2-3 horas')
print(f'  Salida : train_embeddings_raw{suffix}.parquet')
print('=' * 55)

CONFIGURACIÓN
  Modo   : DATASET COMPLETO (~545K samples)
  Tiempo : ~2-3 horas
  Salida : train_embeddings_raw.parquet


## 3. Carga de Datos

In [3]:
train_full = pd.read_parquet(DATA_PROCESSED / 'train.parquet')
test_full  = pd.read_parquet(DATA_PROCESSED / 'test.parquet')

print(f'Train completo : {train_full.shape}')
print(f'Test completo  : {test_full.shape}')

if USE_SUBSET:
    train_size = int(SUBSET_SIZE * 0.8)
    test_size  = SUBSET_SIZE - train_size
    train_df, _ = train_test_split(train_full, train_size=train_size,
                                   stratify=train_full['emotion'], random_state=42)
    test_df,  _ = train_test_split(test_full,  train_size=test_size,
                                   stratify=test_full['emotion'],  random_state=42)
    print(f'\nSubset train : {train_df.shape}')
    print(f'Subset test  : {test_df.shape}')
else:
    train_df = train_full
    test_df  = test_full

print('\n✓ Datos listos')

Train completo : (441127, 28)
Test completo  : (110282, 28)

✓ Datos listos


## 4. Carga del Modelo de Embeddings

In [4]:
# all-MiniLM-L6-v2: 384 dimensiones, rápido y buena calidad semántica
print('Cargando sentence-transformer...')
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f'✓ Modelo cargado: {model.get_sentence_embedding_dimension()} dimensiones')

Cargando sentence-transformer...
✓ Modelo cargado: 384 dimensiones


## 5. Generación de Embeddings — Train

In [5]:
BATCH_SIZE = 10000

def encode_texts(df, label):
    texts = df['text_clean'].fillna('').tolist()
    embeddings = []
    t0 = time.time()
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i+BATCH_SIZE]
        embeddings.append(model.encode(batch, show_progress_bar=False, batch_size=32))
        print(f'  {label}: {min(i+BATCH_SIZE, len(texts)):,}/{len(texts):,}')
    result = np.vstack(embeddings)
    print(f'  ✓ {label} listo en {(time.time()-t0)/60:.1f} min — shape: {result.shape}')
    return result

print('Generando embeddings TRAIN...')
train_emb = encode_texts(train_df, 'Train')

Generando embeddings TRAIN...
  Train: 10,000/441,127
  Train: 20,000/441,127
  Train: 30,000/441,127
  Train: 40,000/441,127
  Train: 50,000/441,127
  Train: 60,000/441,127
  Train: 70,000/441,127
  Train: 80,000/441,127
  Train: 90,000/441,127
  Train: 100,000/441,127
  Train: 110,000/441,127
  Train: 120,000/441,127
  Train: 130,000/441,127
  Train: 140,000/441,127
  Train: 150,000/441,127
  Train: 160,000/441,127
  Train: 170,000/441,127
  Train: 180,000/441,127
  Train: 190,000/441,127
  Train: 200,000/441,127
  Train: 210,000/441,127
  Train: 220,000/441,127
  Train: 230,000/441,127
  Train: 240,000/441,127
  Train: 250,000/441,127
  Train: 260,000/441,127
  Train: 270,000/441,127
  Train: 280,000/441,127
  Train: 290,000/441,127
  Train: 300,000/441,127
  Train: 310,000/441,127
  Train: 320,000/441,127
  Train: 330,000/441,127
  Train: 340,000/441,127
  Train: 350,000/441,127
  Train: 360,000/441,127
  Train: 370,000/441,127
  Train: 380,000/441,127
  Train: 390,000/441,127
  Tr

## 6. Generación de Embeddings — Test

In [6]:
print('Generando embeddings TEST...')
test_emb = encode_texts(test_df, 'Test')

Generando embeddings TEST...
  Test: 10,000/110,282
  Test: 20,000/110,282
  Test: 30,000/110,282
  Test: 40,000/110,282
  Test: 50,000/110,282
  Test: 60,000/110,282
  Test: 70,000/110,282
  Test: 80,000/110,282
  Test: 90,000/110,282
  Test: 100,000/110,282
  Test: 110,000/110,282
  Test: 110,282/110,282
  ✓ Test listo en 16.5 min — shape: (110282, 384)


## 7. Exportación de Embeddings Crudos

Se guardan los 384 dims sin reducción. El notebook 04 determinará el umbral de PCA óptimo
antes de comprometer los embeddings definitivos.

In [7]:
def save_embeddings(emb, df, path):
    emb_df = pd.DataFrame(emb, columns=[f'emb_{i}' for i in range(emb.shape[1])])
    emb_df['emotion'] = df['emotion'].values
    emb_df.to_parquet(path, index=False)
    size_mb = path.stat().st_size / 1e6
    print(f'  ✓ {path.name} — {emb_df.shape} — {size_mb:.0f} MB')

print('Guardando embeddings crudos (384 dims)...')
save_embeddings(train_emb, train_df, DATA_EMBEDDINGS / f'train_embeddings_raw{suffix}.parquet')
save_embeddings(test_emb,  test_df,  DATA_EMBEDDINGS / f'test_embeddings_raw{suffix}.parquet')
print('\n✓ Listo. Continuá con el notebook 04 para calibrar PCA.')

Guardando embeddings crudos (384 dims)...
  ✓ train_embeddings_raw.parquet — (441127, 385) — 903 MB
  ✓ test_embeddings_raw.parquet — (110282, 385) — 250 MB

✓ Listo. Continuá con el notebook 04 para calibrar PCA.
